# DuckPD Advanced Features & New Capabilities Walkthrough

Welcome to the **DuckPD Features Walkthrough**! This notebook demonstrates the newest capabilities of DuckPD on **real-world financial market data** using the [AlphaDojo/dojo_stock_news](https://huggingface.co/datasets/AlphaDojo/dojo_stock_news) dataset (~3.9M articles).

### What you will see:
- **Direct Remote Parquet Scanning**: Query millions of rows in cloud parquet without loading full datasets into Python memory.
- **Vectorized String Accessors (`.str`)**: Clean publisher names, extract headlines, and filter topics lazily.
- **Multi-Table Relational Merges (`merge`)**: Join multi-million row news feeds with ticker metadata tables.
- **Multi-Frame Concatenation (`duckpd.concat`)**: Combine filtered partitions with automatic schema union and null-padding.
- **Extended Reductions**: Compute standard deviation (`std`), variance (`var`), median (`median`), and quantiles (`quantile`).
- **Advanced Multi-Column GroupBy**: Named aggregations across publishers and tickers.
- **Window & Positional Transforms**: Cumulative, rank, difference, rolling, expanding, and shifted analytics with guaranteed ordering.
- **Persistence, Query Plans & Direct Parquet Export**: Reuse materialized intermediates, inspect pushdown, and write without pandas fallback.

## 1. Setup Session & Connect to Remote Parquet

Initialize a DuckPD session with custom memory and execution settings, then lazily scan the 3.9M row dataset hosted on Hugging Face.

In [1]:
import pandas as std_pd

import duckpd as pd

print(f"DuckPD Version: {pd.__version__}")
session = pd.connect(memory_limit="1GB", threads=4)

# Remote dataset from AlphaDojo (~3.9M financial news rows)
DATA_URL = "https://huggingface.co/datasets/AlphaDojo/dojo_stock_news/resolve/main/data.parquet"

# Lazily scan Parquet over HTTP; DuckPD preserves its physical file order automatically.
news_df = session.read_parquet(DATA_URL)

print("Lazy DataFrame created:")
print(f"Columns: {news_df.columns}")
print(f"Session executions so far: {session.execution_count}")

DuckPD Version: 0.1.4
Lazy DataFrame created:
Columns: ('title', 'image', 'ago', 'primarysymbol', 'primarytopic', 'publisher', 'url', 'id', 'imagedomain', 'description', 'primarytopic_url', 'publisher_logo', 'publish_date', 'on_symbol_json', 'symbol', 'source')
Session executions so far: 0


## 2. Vectorized String Accessors (`.str`)

Clean publisher names, compute headline lengths, and flag earnings-related announcements lazily using DuckPD's `.str` accessor methods.

In [2]:
# Perform lazy string feature engineering
enriched_news = news_df.assign(
    publisher_clean=news_df["publisher"].str.strip().str.upper(),
    title_len=news_df["title"].str.len(),
    is_earnings=news_df["title"].str.upper().str.contains("EARNINGS"),
    is_option_activity=news_df["title"].str.contains("Option Activity"),
)

# Inspect a bounded preview pushed down to DuckDB
preview_cols = ["symbol", "publisher_clean", "title_len", "is_earnings", "title"]
enriched_news[preview_cols].head(5)

,title,symbol,publisher_clean,title_len,is_earnings
0,"Dollar Tree CIO Sells 2,500 Shares Valued at $...",WYGC,THE MOTLEY FOOL,53,False
1,Why IonQ Stock Jumped 12.4% This Morning,MSPR,THE MOTLEY FOOL,40,False
2,Tuesday's ETF with Unusual Volume: NFRA,MSPR,BNK INVEST,42,False
3,"Tuesday 9/8 Insider Buying Report: ENOV, OBIO",MSPR,BNK INVEST,46,False
4,Billionaire Stanley Druckenmiller Has 2 Megaca...,MSPR,THE MOTLEY FOOL,127,False


## 3. Multi-Table Relational Merging (`merge`)

Join the multi-million row news dataset with a ticker reference metadata table. The join and predicates are compiled into relational SQL execution.

In [3]:
# Reference table for prominent tech & consumer market cap leaders
ticker_meta = session.from_pandas(
    std_pd.DataFrame(
        {
            "symbol": ["AAPL", "NVDA", "MSFT", "AMZN", "TSLA", "GOOGL"],
            "company_name": [
                "Apple Inc.",
                "NVIDIA Corp.",
                "Microsoft Corp.",
                "Amazon.com Inc.",
                "Tesla Inc.",
                "Alphabet Inc.",
            ],
            "sector": [
                "Technology",
                "Semiconductors",
                "Software",
                "E-Commerce",
                "Automotive",
                "Communication",
            ],
            "market_tier": [
                "Mega Cap",
                "Mega Cap",
                "Mega Cap",
                "Mega Cap",
                "Mega Cap",
                "Mega Cap",
            ],
        }
    )
)

# Merge ticker metadata with news stream and sort by publish_date
news_with_sector = ticker_meta.merge(enriched_news, on="symbol", how="inner").sort_values(
    "publish_date"
)

news_with_sector[
    ["symbol", "company_name", "sector", "publisher_clean", "title_len", "title"]
].head(5)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,symbol,company_name,sector,publisher_clean,title_len,title
0,TSLA,Tesla Inc.,Automotive,THE MOTLEY FOOL,73,Elon Musk Says This Is One of the Biggest Chal...
1,TSLA,Tesla Inc.,Automotive,THE MOTLEY FOOL,81,"If You'd Invested $10,000 in Tesla a Decade Ag..."
2,TSLA,Tesla Inc.,Automotive,THE MOTLEY FOOL,128,Cathie Wood's ARK Bought $40 Million of Nvidia...
3,TSLA,Tesla Inc.,Automotive,THE MOTLEY FOOL,94,"SpaceX Stock Has Plunged 34%. Elon Musk's ""Sec..."
4,TSLA,Tesla Inc.,Automotive,ZACKS,109,"The Zacks Analyst Blog Highlights SpaceX, Tesl..."


## 4. Multi-Frame Concatenation (`duckpd.concat`)

Combine distinct ticker news subsets row-wise with automatic schema union and null-padding.

In [4]:
# Split subsets and enrich one partition with custom category tags
nvda_news = news_with_sector[news_with_sector["symbol"] == "NVDA"].assign(focus_area="AI Hardware")[
    ["symbol", "company_name", "focus_area", "publisher_clean", "title"]
]

tsla_news = news_with_sector[news_with_sector["symbol"] == "TSLA"][
    ["symbol", "company_name", "publisher_clean", "title"]
]

# Concatenate partitions: focus_area will be padded with NULLs for TSLA
combined_stream = pd.concat([nvda_news, tsla_news])
print("Union Columns:", combined_stream.columns)

combined_stream.head(6)

Union Columns: ('symbol', 'company_name', 'focus_area', 'publisher_clean', 'title')


,symbol,company_name,focus_area,publisher_clean,title
0,NVDA,NVIDIA Corp.,AI Hardware,THE MOTLEY FOOL,Prediction: This Is Where Nvidia Stock Will Be...
1,NVDA,NVIDIA Corp.,AI Hardware,THE MOTLEY FOOL,Jensen Huang Told Investors in June to 'Buy at...
2,NVDA,NVIDIA Corp.,AI Hardware,THE MOTLEY FOOL,The S&P 500 Has Fallen in 56% of Septembers Si...
3,NVDA,NVIDIA Corp.,AI Hardware,ZACKS,Is Broadcom (AVGO) Stock a Buy Before Its Q3 E...
4,NVDA,NVIDIA Corp.,AI Hardware,THE MOTLEY FOOL,Nvidia (NVDA) Q2 2027 Earnings Call Transcript
5,NVDA,NVIDIA Corp.,AI Hardware,MARKETBEAT,Broadcom’s Earnings Test Comes With a Higher B...


## 5. Extended Statistical & Boolean Reductions

Calculate statistical metrics across headline length and content properties (`mean`, `median`, `std`, `var`, `quantile`, `any`, `all`) computed in a single SQL query in DuckDB.

In [5]:
print("--- Headline Length Statistical Metrics ---")
print(f"Mean Length:       {news_with_sector['title_len'].mean():.2f}")
print(f"Median Length:     {news_with_sector['title_len'].median():.2f}")
print(f"Std Deviation:     {news_with_sector['title_len'].std():.2f}")
print(f"Variance:          {news_with_sector['title_len'].var():.2f}")
print(f"25th Percentile:   {news_with_sector['title_len'].quantile(0.25):.2f}")
print(f"75th Percentile:   {news_with_sector['title_len'].quantile(0.75):.2f}")
print(f"95th Percentile:   {news_with_sector['title_len'].quantile(0.95):.2f}")

print("\n--- Boolean Reductions on Filtered Subset ---")
print(f"All headlines mention earnings? {news_with_sector['is_earnings'].all()}")
print(f"Any headline mentions earnings? {news_with_sector['is_earnings'].any()}")

--- Headline Length Statistical Metrics ---
Mean Length:       80.75
Median Length:     72.50
Std Deviation:     31.09
Variance:          966.36
25th Percentile:   60.00
75th Percentile:   99.00
95th Percentile:   147.00

--- Boolean Reductions on Filtered Subset ---
All headlines mention earnings? False
Any headline mentions earnings? True


## 6. Advanced GroupBy & Multi-Metric Aggregations

Perform analytical grouping across publishers and tickers using named aggregations, calculating article volume, average length, dispersion, and extreme values.

In [6]:
# Aggregate news analytics by publisher across top market-cap tickers
publisher_analytics = (
    news_with_sector.groupby(["publisher_clean"], as_index=False)
    .agg(
        article_count=("title", "count"),
        avg_headline_len=("title_len", "mean"),
        std_headline_len=("title_len", "std"),
        max_headline_len=("title_len", "max"),
        min_headline_len=("title_len", "min"),
    )
    .sort_values("article_count", ascending=False)
)

publisher_analytics.head(10)

,publisher_clean,article_count,avg_headline_len,std_headline_len,max_headline_len,min_headline_len
0,THE MOTLEY FOOL,772,90.461140,32.940575,204,24
1,ZACKS,250,63.676000,12.118020,109,28
2,MARKETBEAT,66,61.848485,9.867183,81,41
3,BNK INVEST,36,39.277778,10.250048,53,24
4,NASDAQ.COM,27,100.814815,11.903577,111,76
5,RTTNEWS,27,64.888889,13.003944,95,34
6,BARCHART,22,53.227273,9.541547,78,36


## 7. Window & Positional Transforms (`cumsum`, `rank`, `diff`)

Execute analytical window operations over ordered streams. CSV, Parquet, pandas, and Arrow sources carry an automatic ordering guarantee; SQL/table relations and post-join workflows must establish one with `order_by` or `sort_values`.

In [7]:
# Compute cumulative article counts, volume ranks, and incremental step differences
ranked_publishers = publisher_analytics.assign(
    volume_rank=publisher_analytics["article_count"].rank(method="dense", ascending=False),
    cumulative_articles=publisher_analytics["article_count"].cumsum(),
    article_step_diff=publisher_analytics["article_count"].diff(-1),
)

ranked_publishers.head(10)

,publisher_clean,article_count,avg_headline_len,std_headline_len,max_headline_len,min_headline_len,volume_rank,cumulative_articles,article_step_diff
0,THE MOTLEY FOOL,772,90.461140,32.940575,204,24,1.0,772,522.0
1,ZACKS,250,63.676000,12.118020,109,28,2.0,1022,184.0
2,MARKETBEAT,66,61.848485,9.867183,81,41,3.0,1088,30.0
3,BNK INVEST,36,39.277778,10.250048,53,24,4.0,1124,9.0
4,NASDAQ.COM,27,100.814815,11.903577,111,76,5.0,1151,0.0
5,RTTNEWS,27,64.888889,13.003944,95,34,5.0,1178,5.0
6,BARCHART,22,53.227273,9.541547,78,36,6.0,1200,NaN


## 8. Rolling, Expanding & Shifted Analytics

Build richer ordered analytics with row-based rolling and expanding windows. The transforms stay lazy and compile into DuckDB window expressions; persisting then creates a reusable DuckDB table at an explicit execution boundary.

In [8]:
publisher_windows = ranked_publishers.assign(
    rolling_3_avg_articles=ranked_publishers["article_count"].rolling(3, min_periods=1).mean(),
    expanding_articles=ranked_publishers["article_count"].expanding().sum(),
    previous_publisher_articles=ranked_publishers["article_count"].shift(1),
)

persisted_publishers = publisher_windows.persist("publisher_window_summary")
print(f"Executions after persist: {session.execution_count}")
persisted_publishers.head(10)

Executions after persist: 15


,publisher_clean,article_count,avg_headline_len,std_headline_len,max_headline_len,min_headline_len,volume_rank,cumulative_articles,article_step_diff,rolling_3_avg_articles,expanding_articles,previous_publisher_articles
0,THE MOTLEY FOOL,772,90.461140,32.940575,204,24,1.0,772,522.0,772.000000,772.0,NaN
1,ZACKS,250,63.676000,12.118020,109,28,2.0,1022,184.0,511.000000,1022.0,772.0
2,MARKETBEAT,66,61.848485,9.867183,81,41,3.0,1088,30.0,362.666667,1088.0,250.0
3,BNK INVEST,36,39.277778,10.250048,53,24,4.0,1124,9.0,117.333333,1124.0,66.0
4,NASDAQ.COM,27,100.814815,11.903577,111,76,5.0,1151,0.0,43.000000,1151.0,36.0
5,RTTNEWS,27,64.888889,13.003944,95,34,5.0,1178,5.0,30.000000,1178.0,27.0
6,BARCHART,22,53.227273,9.541547,78,36,6.0,1200,NaN,25.333333,1200.0,27.0


## 9. Plan Inspection (`explain()`) & Direct Export

Inspect the relational query plan generated by DuckPD, including predicate pushdown, joins, and window functions. Then write the window-enriched summary directly to Parquet without routing the full result through pandas.

In [9]:
print("=== Compiled Query Plan with Window Transforms ===")
print(publisher_windows.explain())

# Export the analytical summary directly from DuckDB.
publisher_windows.write_parquet("stock_news_summary.parquet", overwrite=True)
print("\nExported stock_news_summary.parquet directly via DuckDB!")

=== Compiled Query Plan with Window Transforms ===


Fallback boundaries: none (policy=error)
Materialization boundaries: none in the logical plan
Remote source boundaries: none
Source fragments: [{"estimated_transfer_bytes": null, "kind": "parquet", "local_required": ["projection", "aggregation", "join", "window", "sort"], "pushdown_candidates": [], "requested": ["projection", "aggregation", "join", "window", "sort"], "source": "https://huggingface.co/datasets/AlphaDojo/dojo_stock_news/resolve/main/data.parquet"}]
Cross-source movement: [{"estimated_transfer_bytes": null, "kind": "cross_source_join", "left": [{"kind": "pandas", "locations": ["a84703386d30461fb6b65588c5d9f6af"]}], "materializes_in_python": false, "right": [{"kind": "parquet", "locations": ["https://huggingface.co/datasets/AlphaDojo/dojo_stock_news/resolve/main/data.parquet"]}], "strategy": "stream_inputs_to_duckdb"}]
Resource policy: {"non_spillable_aggregate_states": "error", "rejected": ["list", "string_agg"]}
Vector operations: []
Embedding operations: []
DuckPD logic